In [ ]:
#Standard packages:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import graphviz
import torch


# Model:

from sklearn.metrics import balanced_accuracy_score
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
import cudf
import lightgbm as lgb
from sklearn.ensemble import AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report



# Data processing:

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA
from statsmodels.stats.outliers_influence import variance_inflation_factor
from imblearn.combine import SMOTETomek

%matplotlib inline
sns.set_style('whitegrid')
%load_ext cudf.pandas
%load_ext cuml.accel

pd.set_option('display.float_format', '{:.10f}'.format)


In [ ]:
# Reading in data
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv').copy()

test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv').copy()

origi = pd.read_csv('/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv').copy()


# Scoring data sets
acc_scores = pd.read_csv('/kaggle/input/datasets/thabangmphela0401/scoring/AccuracyScores.csv').copy()

balanced_acc_scores_unseen = pd.read_csv('/kaggle/input/datasets/thabangmphela0401/scoring/Unseen_balanced_acc.csv').copy()

# Structure of data # 
* No null values found
* More numerical than categorical dtypes
* Imbalanced data set

In [ ]:

train.info()

test.info()
set(train.columns) - set(test.columns)

origi.info()
# No null values found within orginal data set

# Remove ID columns:

train_id = train['id']

test_id = test['id']

train = train.drop(columns = ['id'], axis = 1).copy()

test = test.drop(columns = ['id'], axis = 1).copy()


# EDA # 
Evaluating relationships of categorical columns and target 

In [ ]:
origi.columns
sns.displot(
    origi,
    x = 'Soil_Type',
    col = 'Irrigation_Need',
    height = 5,
    binwidth = 10
)

# Discrete unifrom distribution found from soil types.
# Most crops require low irigiation, followed by medium, lastly high
# Possible inerventions: Change to frequency, fit uniform density over and assign probabilities based on irrigation needed and soil type

sns.displot(
    origi,
    x = 'Crop_Type',
    col = 'Irrigation_Need',
    height=5,
    bins = 30
)


# Similar output to soil type

sns.displot(
    origi,
    x = 'Crop_Growth_Stage',
    col='Irrigation_Need',
    height = 5,
    binwidth = 10
)


# For low irrigation, the graph skews left towards harvest and sowing.
# High + Medium irrigation, graph skews to the right towards vegative and flowering.

sns.displot(
    origi,
    x = 'Season',
    col = 'Irrigation_Need',
    height = 5,
    binwidth = 10
)


# Similar output to soil type and crop type
# Rabi season a bit more frequent in low irrigation


sns.displot(
    origi,
    x = 'Irrigation_Type',
    col = 'Irrigation_Need',
    height = 5,
    binwidth = 10
)


# Canal irrigation is lower for low irrigation but similar to other uniform plots


sns.displot(
    origi,
    x = 'Water_Source',
    col = 'Irrigation_Need',
    height = 5,
    binwidth = 10
)


# Unifrom distribution

sns.displot(
    origi,
    x = 'Mulching_Used',
    col = 'Irrigation_Need',
    height=5,
    binwidth = 10
)


# Low irrigation used alot of Mulching
# Opposite is true for other irrigation needs

sns.displot(
    origi,
    x = 'Region',
    col = 'Irrigation_Need',
    height=5,
    binwidth = 10
)
# North region has a low frequency in low irrigation
# All else uniform distribution

# Data encoding & Feature development

In [ ]:
Xtrain = train.copy()

Ytrain = train['Irrigation_Need'].copy()

Y_Ori = origi['Irrigation_Need'].copy()



Ytrain = Ytrain.map({
    'Low': 0,
    'Medium': 1,
    'High': 2
})

Y_Ori = Y_Ori.map({
    'Low': 0,
    'Medium': 1,
    'High': 2
})

Xtest = test.copy()

Ori = origi.copy()


# Dropping target variable
Xtrain = Xtrain.drop(columns = ['Irrigation_Need','Season'], axis = 1)

Ori = Ori.drop(columns = ['Irrigation_Need','Season'], axis = 1)

Xtest = Xtest.drop(columns = ['Season'], axis = 1)


In [ ]:
# Early feature engineering

def early_fe(df):

    df_new = df.copy()

    # No new features added 


    df_new = df_new.drop(columns = [], axis = 1)

    return df_new

Xtrain_early = early_fe(Xtrain)

Xtest_early = early_fe(Xtest)

Ori_early = early_fe(Ori)

# Encoding variable exploration:



In [ ]:

num_cols = [cname for cname in Xtrain_early.columns
            if pd.api.types.is_integer_dtype(Xtrain_early[cname]) or pd.api.types.is_float_dtype(Xtrain_early[cname])]


cat_cols = list(set(Xtrain_early.columns) - set(num_cols))


# Frequency encoding columns:

freq_cols = ['Region','Crop_Type','Soil_Type']

# Ordinal encoding columns

ordinal_cols = []

ohe_cols = list(set(cat_cols) - set(freq_cols) - set(ordinal_cols) )


# Freqency encoding and ordinal encoding for categorical variables, one hot encoding for the rest of the categorical variables.

def freq_ordi_encoding(df):
    df_new = df.copy()

# Ordinal encoding:


# Frequency encoding:
    for cat in freq_cols:
        freq = pd.concat([Xtrain_early[cat], Xtest_early[cat], Ori_early[cat]]).value_counts()

        df_new[f'freq_{cat}'] = df_new[cat].map(freq).fillna(0).astype('float32')


# Ohe:
    ohe = OneHotEncoder(handle_unknown= 'ignore', sparse_output=False, drop = 'first')

    for var in ohe_cols:
        ohe_df = pd.DataFrame(ohe.fit_transform(df_new[[var]]), columns = [f'{var}_{cat}' for cat in ohe.categories_[0][1:]], index = df.index)
        df_new = pd.merge(df_new, ohe_df, left_index = True, right_index = True)

    cols_to_drop = freq_cols + ohe_cols + ordinal_cols

    df_new = df_new.drop(columns = cols_to_drop, axis = 1)


    return df_new

Xtrain_encoded = freq_ordi_encoding(Xtrain_early)


Xtest_encoded = freq_ordi_encoding(Xtest_early)


Ori_encoded = freq_ordi_encoding(Ori_early)



In [ ]:
# understanding variation within features 

Xtrain_encoded.select_dtypes(include=('float')).agg('var').sort_values(ascending=False)


In [ ]:


def feature_engineering(df):
    df_new = df.copy()

    # Numerical feature engineering

    # Rainfall_mm

    # High variation within rainfall - highlights outliers
    df_new['Outliers_Rainfall'] = np.absolute(df_new['Rainfall_mm'] - df_new['Rainfall_mm'].mean()) # Champion of a feature

    # A feature that adds direction to deviance of rainfall - helps imbalanced 'high' class
    df_new['Low_Rain'] = (
        df_new['Rainfall_mm'] < np.quantile(df_new['Rainfall_mm'], 0.25)
    ).astype(int)


   # Previous Irrigation

    df_new['Outliers_prev_irrigation'] =np.absolute(df_new['Previous_Irrigation_mm'] - df_new['Previous_Irrigation_mm'].mean())

    # Humidity

    df_new['Outliers_humidity'] = np.absolute(df_new['Humidity'] - df_new['Humidity'].mean())

    # Soil moisture
    df_new['Moisture_Level'] = (np.log(df_new['Soil_Moisture']) + np.log(df_new['Rainfall_mm']))/(np.log(df_new['Temperature_C']) + np.log(df_new['Wind_Speed_kmh'] - df_new['Wind_Speed_kmh'].min() + 1))

    df_new['Moisture_Level'] = (
        df_new['Moisture_Level'] < np.quantile(df_new['Moisture_Level'], 0.25)
    ).astype(int)

    # Numerical columns commmented out were left due to high correaltion

    # Soil pH
    #df_new['High_Soil_ph'] = (
        #df_new['Soil_pH'] > np.quantile(df_new['Soil_pH'], 0.75)
    #).astype(int)

    # Temperature

    #df_new['Outliers_temp'] = np.absolute(df_new['Temperature_C'] - df_new['Temperature_C'].mean())

    # Field Area hectare
    #df_new['Large_Field'] = (
        #df_new['Field_Area_hectare'] > np.quantile(df_new['Field_Area_hectare'], 0.75)
    #).astype(int)

    # Organic Carbon outliers

    #df_new['Organic_Carbon_outliers'] = np.absolute(df_new['Organic_Carbon'] - df_new['Organic_Carbon'].mean())

    # Electrical conductivity

    #df_new['Electrical_Conductivity_outliers'] = np.absolute(df_new['Electrical_Conductivity'] - df_new['Electrical_Conductivity'].mean())

    # Temperature + EC + OC + Wind Speed

    # Wind speed


    # Adding decimals from numeric cols

    df_new['Rainfall_mm_decimals'] = np.round(df_new['Rainfall_mm'].astype('float64'), decimals=10) - np.round(df_new['Rainfall_mm'].astype('float64'), decimals=0)


    cols_to_drop =  list(['Rainfall_mm'])

    df_new = df_new.drop(columns = cols_to_drop, axis = 1)

    return df_new


Xtrain_fe = feature_engineering(Xtrain_encoded)

Xtest_fe = feature_engineering(Xtest_encoded)

Ori_fe = feature_engineering(Ori_encoded)



In [ ]:
# Binnning:

# No numerical binning that improved balanced accuracy score 



def binner(df, bins, labs, col):


    df_new = df.copy()


    df_new[f'{col}_binned'] = pd.qcut(df_new[col], bins, labels = labs).astype('int')


    df_new = df_new.drop(columns = col, axis = 1)


    return df_new


Xtrain_bins = Xtrain_fe 
Xtest_bins = Xtest_fe 
Ori_bins = Ori_fe 




In [ ]:
# Normalizing:

cols_to_norm = list(set(Xtrain_bins.columns) - set(['Outliers_Rainfall','Rainfall_mm_decimals','Soil_moisture-hectare']))

def nor(df):

    df_new = df.copy()
    minx = np.min(df_new[cols_to_norm], axis = 0)

    maxx = np.max(df_new[cols_to_norm], axis = 0)

    df_new[cols_to_norm] = (df_new[cols_to_norm] - minx)/(maxx - minx)

    return df_new

Xtrain_normalized = nor(Xtrain_bins)

Xtest_normalized = nor(Xtest_bins)

Ori_normalized = nor(Ori_bins)


In [ ]:
# Correlation matrix:

cormat = Xtrain_normalized.corr()


plt.figure(figsize=(30,30))
sns.heatmap(
    cormat,
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

In [ ]:
# Vif scores:

vif_scores = pd.DataFrame()

vif_scores['Feature'] = Xtrain_normalized.columns

vif_scores['Scores'] = [variance_inflation_factor(Xtrain_normalized.values, i) for i in range(Xtrain_normalized.shape[1])]

print(vif_scores.sort_values(by='Scores')) # Vif scores < 6 were used since boosting models aren't effected much by multicollinearity

In [ ]:
# Device agnostic code

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

# Model :


In [ ]:
# Grid search for xgboost model:


skf = StratifiedKFold(
    random_state=42,
    shuffle=True,
    n_splits=5
)

xgb_mod = xgb.XGBClassifier(
    random_state=42,
    objective='multi:softmax',
    num_class=3,
    n_jobs = 1,
    device = 'cuda',
    tree_method = 'hist'
)

param_grid = {
    'n_estimators': [200, 300, 400, 500],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.1, 1]

}


grid_search = GridSearchCV(
    estimator=xgb_mod,
    param_grid=param_grid,
    n_jobs=-1,
    scoring='balanced_accuracy',
    cv=skf,
    return_train_score=False,
    verbose=1
)


grid_search.fit(Xtrain_normalized, Ytrain)


In [ ]:
# Results:
best_parms = str(grid_search.best_params_)

best_acc = grid_search.best_score_

model_no = 73
acc_scores.loc[model_no, 'Model-no:'] =  model_no

acc_scores.loc[model_no, 'Parameters'] = str(best_parms)

acc_scores.loc[model_no, 'Accuracy_Score'] = best_acc

acc_scores.to_csv('AccuracyScores.csv', index=False)



In [ ]:
acc_scores.tail()

# Evaluating model performance on unseen data set: #

In [ ]:


y_preds_orig = grid_search.predict(Ori_normalized)

balanced_acc_scores_unseen.loc[model_no, 'Model_no'] = model_no

balanced_acc_scores_unseen.loc[model_no, 'Score'] = balanced_accuracy_score(Y_Ori, y_preds_orig)

balanced_acc_scores_unseen.loc[model_no, 'Model_Change'] = str('Changes made: Combin of stage and crop type')

balanced_acc_scores_unseen.to_csv('Unseen_balanced_acc.csv', index=False)

In [ ]:
balanced_acc_scores_unseen.tail()


In [ ]:
# Confustion matrix:

plt.figure(figsize = (5,4))
cm = confusion_matrix(y_preds_orig, Y_Ori)

sns.heatmap(
    cm,
    annot=True,
    fmt = 'd',
    cmap='Blues'
)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [ ]:
# Feature importance:

plt.figure(figsize = (8,6))
xgb.plot_importance(grid_search.best_estimator_)
plt.title('Feature Importance')


In [ ]:
# Classification report
cr = classification_report(Y_Ori, y_preds_orig)
print(cr)

In [ ]:
# Tree

plt.figure(figsize = (100,50))
xgb.plot_tree(grid_search.best_estimator_, num_trees=0)
plt.savefig('tree.png', dpi=300, bbox_inches='tight')


In [ ]:
# Final xgboost model :


xgboost_model = xgb.XGBClassifier(
    random_state=42,
    objective='multi:softmax',
    num_class=3,
    n_jobs = 1,
    learning_rate = 1,
    n_estimators = 200,
    max_depth = 3
)

xgboost_model.fit(Xtrain_normalized, Ytrain)

ytrain_xgb = xgboost_model.predict(Ori_normalized)

ypreds_xgb = xgboost_model.predict(Xtest_normalized)



In [ ]:
# Light gbm  classifier:

skf = StratifiedKFold(
    random_state=42,
    shuffle=True,
    n_splits=5
)

lgb_mod = lgb.LGBMClassifier(
     objective='multiclass',
    device='gpu',              
    gpu_platform_id=0,       
    gpu_device_id=0,           
    n_jobs=1,                  
    force_row_wise=True 
)

param_grid_lgb = {
    'n_estimators' : [200, 300],
    'max_depth' : [3,5]

}

grid_search_lgb = GridSearchCV(
    estimator=lgb_mod,
    param_grid= param_grid_lgb,
    cv=skf,
    return_train_score=False,
    verbose=0,
    scoring='balanced_accuracy'
)

grid_search_lgb.fit(Xtrain_normalized, Ytrain)

In [ ]:
# Results:
best_parms_lgb = str(grid_search_lgb.best_params_)

best_acc_lgb = grid_search_lgb.best_score_

model_no = 1

acc_scores_lgb = pd.DataFrame(columns = ['Model-no','Parameters','Accuracy_Score'])

acc_scores_lgb.loc[model_no, 'Model-no:'] =  model_no

acc_scores_lgb.loc[model_no, 'Parameters'] = str(best_parms_lgb)

acc_scores_lgb.loc[model_no, 'Accuracy_Score'] = best_acc_lgb

acc_scores_lgb.to_csv('AccuracyScores_lgb.csv', index=False)

In [ ]:
# Comparing light gbm to xgboost
y_preds_orig_xgb = grid_search.predict(Ori_normalized)

y_preds_orig_lgb = grid_search_lgb.predict(Ori_normalized)


In [ ]:
# Final lgb model

lgb_model = lgb.LGBMClassifier(
     objective='multiclass',
    device='gpu',              
    gpu_platform_id=0,         
    gpu_device_id=0,           
    n_jobs=1,                  
    force_row_wise=True,
    n_estimators = 300,
    max_depth = 5
)

lgb_model.fit(Xtrain_normalized, Ytrain)

ytrain_lgb = lgb_model.predict(Ori_normalized)

ypreds_lgb = lgb_model.predict(Xtest_normalized)

In [ ]:
(y_preds_orig_xgb != y_preds_orig_lgb).sum()


In [ ]:
# Catboost model:

skf = StratifiedKFold(
    random_state = 42,
    shuffle = True,
    n_splits = 5
)

Score_card_ctb = pd.DataFrame(columns = ['Fold','Score'])


for i, (train_idx, val_idx) in enumerate(skf.split(Xtrain_normalized,Ytrain)):
    
    print(f'Training Fold_{i}')

    X_train, X_test = Xtrain_normalized.iloc[train_idx], Xtrain_normalized.iloc[val_idx]

    y_train, y_test = Ytrain.iloc[train_idx], Ytrain.iloc[val_idx]

    ctb = CatBoostClassifier(
        n_estimators = 200,
        learning_rate = 0.1,
        max_depth = 3,
        verbose = 0
    )

    ctb.fit(X_train, y_train)

    y_preds = ctb.predict(X_test)


    scoring = balanced_accuracy_score(y_test, y_preds)

    Score_card_ctb.loc[i, 'Fold'] = i
    Score_card_ctb.loc[i, 'Score'] = scoring




In [ ]:
Score_card_ctb

In [ ]:
# Final catboost model

ctb_model = CatBoostClassifier(
        n_estimators = 200,
        learning_rate = 0.1,
        max_depth = 3,
        verbose = 0
    )

ctb_model.fit(Xtrain_normalized, Ytrain)

ytrain_ctb = ctb_model.predict_proba(Ori_normalized)

ypreds_ctb = ctb_model.predict_proba(Xtest_normalized)

In [ ]:
np.round(ytrain_ctb, decimals = 2)

In [ ]:
# Method to try adjust the recall score for class 2

df = pd.DataFrame(ytrain_ctb)

threshold = 0.4

max_probs = df.max(axis=1)
pred_classes = df.idxmax(axis = 1)

result = np.where(max_probs >= threshold, pred_classes, 2)
result = pd.DataFrame(result, columns = ['preds'])

In [ ]:
pred_classes

In [ ]:
print(result)

In [ ]:
# Saving our models

ctb_model.save_model('catboost_model_irrigation_need_comp.cbm')
xgboost_model.save_model('xgboost_model_irrigation_need_comp.json')
lgb_model.booster_.save_model('lgboost_model_irrigation_need_comp.txt')


In [ ]:
# Training data for meta-model
meta_data_train = pd.concat([pd.DataFrame(ytrain_ctb, columns = ['Y1']), pd.DataFrame(ytrain_lgb, columns = ['Y2']), pd.DataFrame(y_preds_orig_xgb, columns = ['Y3'])], axis = 1)

meta_data_final = pd.concat([pd.DataFrame(ypreds_ctb, columns = ['Y1']), pd.DataFrame(ypreds_lgb, columns = ['Y2']), pd.DataFrame(ypreds_xgb, columns = ['Y3'])], axis = 1)

meta_data_final

In [ ]:
print(meta_data_final.sum(axis=1))

In [ ]:
# Linear regression

lgr = LogisticRegression()

lgr.fit(meta_data_train, Y_Ori)

final_preds = lgr.predict(meta_data_final)

In [ ]:
final_preds = pd.DataFrame(final_preds, columns = ['Irrigation_Need'])

final_preds.info()

In [ ]:
mappings = {
    0:'Low',
    1:'Medium',
    2:'High'
}

final_preds = final_preds['Irrigation_Need'].map(mappings)

final_preds

# Final evaluation: # 
* Stacking method managed to produce an accuracy score of 0.96 
* Model performed poorly on 3 class due to imbalance in data set
* Below a new method was applied late in the competition to try improve accuracy 

In [ ]:
# Stacking classifier method

estimators = [
    ('lgb', lgb.LGBMClassifier(
     objective='multiclass',           
    n_jobs=1,                  
    force_row_wise=True,
    n_estimators = 300,
    max_depth = 5
)),
    ('xgb', xgb.XGBClassifier(
    random_state=42,
    objective='multi:softmax',
    num_class=3,
    n_jobs = 1,
    learning_rate = 1,
    n_estimators = 200,
    max_depth = 3
)),
    ('mlp', MLPClassifier(
    hidden_layer_sizes = (100,),
    activation = 'relu',
    solver = 'adam',
    learning_rate = 'constant',
    learning_rate_init = 0.1,
    max_iter = 300
))
    
]

In [ ]:
clf = StackingClassifier(
    estimators = estimators,
    final_estimator = SVC(),
    cv = 5,
    verbose = 0
)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(Xtrain_normalized, Ytrain,
                                                   stratify = Ytrain,
                                                   random_state = 42)

clf.fit(X_train,y_train).score(X_test, y_test)

In [ ]:
Xtrain_final = pd.concat([Xtrain_normalized,Ori_normalized], axis = 0)
ytrain_final = pd.concat([Ytrain,Y_Ori], axis = 0)

In [ ]:
# Final predictions for stacking classifier:

clf.fit(Xtrain_final, ytrain_final)

final_preds_clf = clf.predict(Xtest_normalized)

In [ ]:
final_preds_clf = pd.DataFrame(final_preds_clf, columns = ['Irrigation_Need'])

mappings = {
    0:'Low',
    1:'Medium',
    2:'High'
}

final_preds_clf = final_preds_clf['Irrigation_Need'].map(mappings)

final_preds_clf

In [ ]:
# Exporting data

#submission = pd.concat([test_id,final_preds], axis = 1)

submission_clf = pd.concat([test_id,final_preds_clf], axis = 1)

submission_clf

In [ ]:
submission.to_csv('Submission2.csv', index = False)

In [ ]:
submission_clf.to_csv('Submission1_clf.csv', index = False)